# RibFrac 预处理（Colab）

与 `colab_tcia_covid19_preprocess.ipynb`、`preprocess/proc_spatial_sequence.py` **相同的处理逻辑**：
- 每个 nii volume：**50 个 npy**（轴向 z≥128 时）或 **33 个 npy**（z<128 时，`int(50/1.5)`）
- patch 大小 128³，`Resize` + `ScaleIntensityRangePercentiles(1, 99.9)`

**数据**：在 `unzip/` 下递归查找，**仅处理** `*-image.nii.gz`（CT）；跳过 `*-label.nii.gz`，与 TCIA 仅对体积做 spatial patch 预训练预处理一致。

**输出**：预处理得到的 `.npy` 写入与 `unzip/` **平级**的 `npy/` 目录。

In [2]:
# Cell 1: 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Cell 2: 安装依赖
!pip install -q nibabel monai

In [ ]:
# Cell 3: RibFrac 预处理（与 TCIA 笔记本 / proc_spatial_sequence 逻辑一致）
import os
import random
import numpy as np
import nibabel as nib
from monai.transforms import Compose, Resize, ScaleIntensityRangePercentiles
from tqdm import tqdm

# ========== 配置：RibFrac 解压目录与输出目录 ==========
# 与 download 同级的 unzip；与 unzip 平级的 npy 存放 patch 的 .npy
DRIVE_RIBFRAC_UNZIP = '/content/drive/MyDrive/dataset/pretrain/RibFrac/unzip'
DRIVE_SAVE_ROOT = '/content/drive/MyDrive/dataset/pretrain/RibFrac/npy'

# ========== 与 proc_spatial_sequence.py 完全一致 ==========
PATCH_NUM = 50
PATCH_SIZE_LIST = [(128, 128, 128)]
TAR_IMG_SIZE = [128, 128, 128]
START_INDEX = 0


def cut_patch(img_array, patch_size):
    img_shape = img_array.shape
    size_z, size_y, size_x = patch_size
    center_x_min, center_x_max = size_x // 2, img_shape[2] - size_x // 2 - 1
    center_y_min, center_y_max = size_y // 2, img_shape[1] - size_y // 2 - 1
    center_z_min, center_z_max = size_z // 2, img_shape[0] - size_z // 2 - 1
    if center_x_min >= center_x_max:
        x1, x2 = 0, img_shape[2]
    else:
        center_x = random.randint(center_x_min, center_x_max)
        x1, x2 = center_x - size_x // 2, center_x + size_x // 2
    if center_y_min >= center_y_max:
        y1, y2 = 0, img_shape[1]
    else:
        center_y = random.randint(center_y_min, center_y_max)
        y1, y2 = center_y - size_y // 2, center_y + size_y // 2
    if center_z_min >= center_z_max:
        z1, z2 = 0, img_shape[0]
    else:
        center_z = random.randint(center_z_min, center_z_max)
        z1, z2 = center_z - size_z // 2, center_z + size_z // 2
    img_patch = img_array[z1:z2, y1:y2, x1:x2]
    return img_patch, [z1, z2, y1, y2, x1, x2]


def load_nii_data(nii_file):
    nii_data = nib.load(nii_file)
    return nii_data.get_fdata(), nii_data.affine


def load_and_patch_transforms(img, tar_img_size):
    transforms = Compose([
        Resize(spatial_size=(tar_img_size[0], tar_img_size[1], tar_img_size[2]), mode='trilinear'),
        ScaleIntensityRangePercentiles(lower=1., upper=99.9, b_min=0.0, b_max=1.0, clip=True, relative=False, channel_wise=False),
    ])
    return transforms(img)


def cut_and_save_one_volume(image_file, patch_size_list, patch_num, save_root, start_index, tar_img_size):
    image, affine = load_nii_data(image_file)
    path_parts = image_file.replace('\\', '/').split('/')
    ds_name = path_parts[-3] if len(path_parts) >= 3 else 'RibFrac'
    nii_id = path_parts[-1].split('.nii.gz')[0].split('.nii')[0].split('.mha')[0]

    if len(image.shape) == 4:
        return []
    images = [image]
    n, patch_path_list = 0, []

    for image_index, image in enumerate(images):
        image = image.transpose((2, 1, 0))
        _patch_num = int(patch_num / 1.5) if image.shape[0] < patch_size_list[0][2] else patch_num
        for i in range(_patch_num):
            n += 1
            patch_size = random.choice(patch_size_list)
            image_patch, _ = cut_patch(image, patch_size)
            image_patch = image_patch.transpose((2, 1, 0))
            image_patch = load_and_patch_transforms(np.expand_dims(image_patch, 0), tar_img_size).numpy()[0, ...]
            save_name = os.path.join(save_root, '%s_%s_%s_%d.nii.gz' % (ds_name, nii_id, image_index, start_index + n))
            patch_path_list.append(save_name)
            np.save(save_name.replace('.nii.gz', '.npy'), image_patch)
    return patch_path_list


def collect_ribfrac_image_nii_files(base_path):
    """仅 *-image.nii.gz，与 TCIA 仅体积预处理一致；跳过 label。"""
    nii_files = []
    for root, _, files in os.walk(base_path):
        for f in files:
            if not f.endswith('.nii.gz') and not f.endswith('.nii'):
                continue
            if '-label.nii.gz' in f or f.endswith('-label.nii'):
                continue
            if '-image.nii.gz' in f or f.endswith('-image.nii'):
                nii_files.append(os.path.join(root, f))
    return sorted(nii_files)


os.makedirs(DRIVE_SAVE_ROOT, exist_ok=True)
data_list = collect_ribfrac_image_nii_files(DRIVE_RIBFRAC_UNZIP)
print(f'找到 {len(data_list)} 个 RibFrac 图像 nii 文件（已排除 label）')

if len(data_list) == 0:
    print(f'未找到文件，请检查: {DRIVE_RIBFRAC_UNZIP}')
else:
    patch_list_all = []
    for i, path in enumerate(tqdm(data_list, desc='预处理')):
        patch_list = cut_and_save_one_volume(path, PATCH_SIZE_LIST, PATCH_NUM, DRIVE_SAVE_ROOT, START_INDEX, TAR_IMG_SIZE)
        patch_list_all += patch_list
        if (i + 1) % 20 == 0:
            print(f'[{i+1}/{len(data_list)}] {path} -> {len(patch_list)} npy')
    print(f'\n完成! 共 {len(patch_list_all)} 个 npy, 输出: {DRIVE_SAVE_ROOT}')

In [ ]:
# Cell 4 (可选): 生成 train_patch_spatial.txt 用于 pretrain
list_save_dir = os.path.join(os.path.dirname(DRIVE_SAVE_ROOT), 'pretrain_lists')
os.makedirs(list_save_dir, exist_ok=True)
npy_files = sorted([os.path.join(DRIVE_SAVE_ROOT, f) for f in os.listdir(DRIVE_SAVE_ROOT) if f.endswith('.npy')])
with open(os.path.join(list_save_dir, 'train_patch_spatial.txt'), 'w') as f:
    f.write('\n'.join(npy_files))
print(f'train_patch_spatial.txt 已保存: {list_save_dir}')

In [4]:
# RibFrac 预处理完成度校验（与 colab_ribfrac_preprocess 逻辑一致）
import os
import nibabel as nib
from tqdm import tqdm

DRIVE_RIBFRAC_UNZIP = "/content/drive/MyDrive/dataset/pretrain/RibFrac/unzip"
DRIVE_NPY_ROOT = "/content/drive/MyDrive/dataset/pretrain/RibFrac/npy"
START_INDEX = 0  # 若预处理里改过 START_INDEX，请改成相同值
PATCH_NUM = 50
PATCH_Z_THR = 128  # image.transpose(2,1,0) 后的 shape[0] 与此比较


def collect_ribfrac_image_nii_files(base_path):
    out = []
    for root, _, files in os.walk(base_path):
        for f in files:
            if not f.endswith(".nii.gz") and not f.endswith(".nii"):
                continue
            if "-label.nii.gz" in f or f.endswith("-label.nii"):
                continue
            if "-image.nii.gz" in f or f.endswith("-image.nii"):
                out.append(os.path.join(root, f))
    return sorted(out)


def meta_from_path(image_file):
    parts = image_file.replace("\\", "/").split("/")
    ds_name = parts[-3] if len(parts) >= 3 else "RibFrac"
    nii_id = parts[-1].split(".nii.gz")[0].split(".nii")[0].split(".mha")[0]
    return ds_name, nii_id


def z_len_after_transpose(nii_path):
    img = nib.load(nii_path).get_fdata()
    if len(img.shape) == 4:
        return None
    return img.transpose((2, 1, 0)).shape[0]


def expected_patches(z0):
    if z0 is None:
        return 0
    return int(PATCH_NUM / 1.5) if z0 < PATCH_Z_THR else PATCH_NUM


def parse_npy_stem(stem):
    # {ds_name}_{nii_id}_{image_index}_{n} —— nii_id / ds_name 中不含下划线时用 rsplit 即可
    a, b, c, d = stem.rsplit("_", 3)
    return a, b, int(c), int(d)


# ---------- 扫描体积 ----------
nii_paths = collect_ribfrac_image_nii_files(DRIVE_RIBFRAC_UNZIP)
expected = {}  # (ds, nii_id) -> dict
for p in tqdm(nii_paths, desc="读 nii 推导应有 patch 数"):
    ds, nid = meta_from_path(p)
    z0 = z_len_after_transpose(p)
    exp = expected_patches(z0)
    key = (ds, nid)
    expected[key] = {
        "path": p,
        "z_after_T": z0,
        "n_patches": exp,
        "skipped_4d": z0 is None,
    }

# ---------- 扫描 npy，按 (ds, nii_id) 统计 ----------
actual_counts = {k: set() for k in expected}
orphan_files = []
if not os.path.isdir(DRIVE_NPY_ROOT):
    print(f"缺少目录: {DRIVE_NPY_ROOT}")
else:
    for fn in os.listdir(DRIVE_NPY_ROOT):
        if not fn.endswith(".npy"):
            continue
        stem = fn[:-4]
        try:
            ds, nid, img_idx, n = parse_npy_stem(stem)
        except ValueError:
            orphan_files.append(fn)
            continue
        key = (ds, nid)
        if key not in actual_counts:
            orphan_files.append(fn)
            continue
        if img_idx != 0:
            orphan_files.append(fn)
            continue
        lo, hi = START_INDEX + 1, START_INDEX + expected[key]["n_patches"]
        if not (lo <= n <= hi):
            orphan_files.append(fn)
            continue
        actual_counts[key].add(n)

# ---------- 汇总 ----------
bad = []
missing_idx = []
for key, info in expected.items():
    if info["skipped_4d"]:
        bad.append((key, "4D 已跳过，不应有 patch", info))
        continue
    exp_n = info["n_patches"]
    got = len(actual_counts[key])
    needed = set(range(START_INDEX + 1, START_INDEX + exp_n + 1))
    if actual_counts[key] != needed:
        bad.append(
            (
                key,
                f"应有 {exp_n} 个索引 {START_INDEX+1}..{START_INDEX+exp_n}，实际 {got} 个",
                info,
            )
        )

print("=" * 72)
print(f"unzip 下 *-image.nii.gz 数量: {len(expected)}")
print(f"npy 目录下 .npy 数量: {len([f for f in os.listdir(DRIVE_NPY_ROOT) if f.endswith('.npy')]) if os.path.isdir(DRIVE_NPY_ROOT) else 0}")
print(f"无法匹配的 .npy（或索引异常）: {len(orphan_files)}")
if orphan_files[:15]:
    print("  样例:", orphan_files[:15])
print(f"体积 patch 不齐或与预期不符: {len(bad)}")
if bad[:20]:
    for key, msg, info in bad[:20]:
        print(f"  - {key}: {msg}")
        print(f"    {info['path']}")
if len(bad) > 20:
    print(f"  ... 另有 {len(bad) - 20} 条")
print("=" * 72)

if not bad and not orphan_files:
    print("结论: 与当前逻辑对照，预处理已全部对齐（体积数、每例 patch 编号范围一致）。")
else:
    print("结论: 存在问题，请根据上述列表排查（未跑完、路径不一致、或改过 START_INDEX/PATCH_NUM）。")

# ---------- 可选：train_patch_spatial.txt 行数是否等于 npy 数 ----------
list_path = "/content/drive/MyDrive/dataset/pretrain/RibFrac/pretrain_lists/train_patch_spatial.txt"
if os.path.isfile(list_path):
    with open(list_path, encoding="utf-8") as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    npy_on_disk = len([f for f in os.listdir(DRIVE_NPY_ROOT) if f.endswith(".npy")]) if os.path.isdir(DRIVE_NPY_ROOT) else 0
    ok_paths = sum(1 for ln in lines if os.path.isfile(ln))
    print(f"\n列表: {list_path}")
    print(f"  非空行数: {len(lines)} | 磁盘上 .npy 数: {npy_on_disk} | 列表中指到且存在的文件: {ok_paths}")
    if len(lines) != npy_on_disk or ok_paths != npy_on_disk:
        print("  提示: 列表与磁盘不一致时，请在生成列表后重跑第 4 格或删掉过时 npy 后重生成。")
else:
    print(f"\n未找到列表文件（若未跑第 4 格可忽略）: {list_path}")

读 nii 推导应有 patch 数: 100%|██████████| 660/660 [52:51<00:00,  4.81s/it]


unzip 下 *-image.nii.gz 数量: 660
npy 目录下 .npy 数量: 32898
无法匹配的 .npy（或索引异常）: 0
体积 patch 不齐或与预期不符: 0
结论: 与当前逻辑对照，预处理已全部对齐（体积数、每例 patch 编号范围一致）。

未找到列表文件（若未跑第 4 格可忽略）: /content/drive/MyDrive/dataset/pretrain/RibFrac/pretrain_lists/train_patch_spatial.txt
